In [1]:
import pandas as pd
import numpy as np

In [2]:
list_all_tokens = np.load('/media/alex/D/Uni/northeastern/data/jlens/list_all_tokens.npy')

In [3]:
import re
from difflib import SequenceMatcher

# 2. seeds
#
# LEX_SEEDS feeds difflib, so it is deliberately conservative: every short or
# English-homographic form is left out. Including them cost 7k false positives
# ('le', 'alt', 'sol', 'sus', 'len', 'opp', 'bal', 'der', 'port' all fire on
# ordinary code tokens; 'linke'/'linker'/'turun'/'onder'/'runter' pull in
# linked, line, turn, wonder, router at ratio .83-.91).
LEX_SEEDS = {
    "up": ["up", "upward", "upwards", "arriba", "subir", "dessus", "monter",
           "oben", "aufwaerts", "aufwärts", "sopra", "acima", "boven", "omhoog",
           "nahoru", "yukari", "yukarı", "epano", "πάνω", "вверх", "вгору", "ऊपर"],
    "down": ["down", "downward", "downwards", "abajo", "abaixo", "bajar",
             "dessous", "descendre", "unten", "herunter", "abwaerts", "abwärts",
             "beneden", "omlaag", "asagi", "aşağı", "κάτω", "вниз", "नीचे"],
    "left": ["left", "leftward", "leftwards", "izquierda", "izquierdo", "gauche",
             "sinistra", "sinistro", "esquerda", "esquerdo", "vanster", "vänster",
             "vasen", "doleva", "vlevo", "stanga", "stânga", "balra", "aristera",
             "αριστερά", "влево", "слева", "ліворуч", "बाएं"],
    "right": ["right", "rightward", "rightwards", "derecha", "derecho", "droite",
              "rechts", "rechte", "destra", "destro", "direita", "direito",
              "hoger", "höger", "oikea", "prawo", "prawa", "vpravo", "doprava",
              "dreapta", "jobbra", "deksia", "δεξιά", "вправо", "справа", "दाएं"],
}

# SEM_ANCHORS feeds the embedder, which scores meaning rather than characters,
# so ambiguous *spellings* are safe here. Ambiguous *meanings* are not, and
# auditing which anchor admitted which token caught four bad ones:
#
#   góra  -> Polish "up" AND "mountain": admitted berg, hillside, Everest,
#            terrain, slope, altitude, Vertices -- 12/12 false positives
#   prawo -> Polish "right" AND "law":   admitted law, LAW, Law, ley, legal (19)
#   links -> German "left" AND English "hyperlink": admitted link, /link, (link
#   sol   -> Turkish "left" AND sun / musical note: admitted '(sol', '-sol'
#
# Each is replaced by a phrase that pins the directional sense. Arrows now go to
# all four directions too: with only ← and → present, '↑' had no correct home
# and was landing under RIGHT.
SEM_ANCHORS = {
    "up":    ["up", "upward", "above", "arriba", "haut", "oben", "sopra", "cima",
              "boven", "upp", "w górę", "вверх", "yukarı", "上", "위", "atas", "↑"],
    "down":  ["down", "downward", "below", "abajo", "bas", "unten", "giù", "baixo",
              "beneden", "ner", "w dół", "вниз", "aşağı", "下", "아래", "bawah", "↓"],
    "left":  ["left", "izquierda", "gauche", "nach links", "sinistra", "esquerda",
              "vänster", "w lewo", "влево", "sol taraf", "左", "왼쪽", "kiri", "←"],
    "right": ["right", "derecha", "droite", "rechts", "destra", "direita",
              "höger", "w prawo", "вправо", "sağ", "右", "오른쪽", "kanan", "→"],
}

print({d: len(v) for d, v in LEX_SEEDS.items()},
      {d: len(v) for d, v in SEM_ANCHORS.items()})

{'up': 22, 'down': 19, 'left': 24, 'right': 26} {'up': 17, 'down': 17, 'left': 14, 'right': 14}


In [4]:
# 3. token -> word pieces
SEP_RE = re.compile(r"""[_\-./\\:,;()\[\]{}<>"'`|!?*+=#@$%^&~\s]+""")
CAMEL_RE = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")


def strip_token(tok):
    """Surface form with whitespace/punctuation shaved off: '_LEFT' -> 'LEFT'."""
    return SEP_RE.sub(" ", tok).strip()


def parts_of(tok):
    """Word pieces to match on, lowercased: '.moveLeft' -> ['move', 'left', 'moveleft']."""
    out = []
    for piece in SEP_RE.split(tok.strip()):
        if piece:
            out.extend(p for p in CAMEL_RE.split(piece) if p)
    whole = SEP_RE.sub("", tok.strip())
    if whole:
        out.append(whole)
    return list(dict.fromkeys(p.lower() for p in out))


parts_of("_LEFT"), parts_of(" moveRight"), parts_of("/right")

(['left'], ['move', 'right', 'moveright'], ['right'])

In [5]:
# 4. lexical stage: difflib over the word pieces
#
# Three guards, all needed:
#
# (a) the ratio cutoff scales with seed length. .80 between 5-char strings just
#     means "one char differs" (light/right, line/linke, turn/turun); the same
#     .62 between 9-char strings is real evidence.
#
# (b) raw ratio cannot separate ischierda/izquierda (.667) from
#     counter/herunter (.667) -- identical scores, and no length or prefix
#     guard splits them. What splits them is that the junk side is always a
#     real English word that needs no direction to explain it, while a mangled
#     foreign direction token is not in the dictionary. So English parts must
#     match essentially exactly; everything else gets the tiered cutoff.
#
# (c) the permissive .62 tier also needs a long *part*, or subword fragments
#     (init, iter, entr, stra, aber, herr) drift into the long foreign seeds.
ENGLISH = {w.strip().lower()
           for w in open("/usr/share/dict/american-english", encoding="latin-1")
           if w.strip()}

flat_lex = [(w, d) for d, ws in LEX_SEEDS.items() for w in ws]


def min_ratio(part, seed):
    if part in ENGLISH:
        return 0.95               # an English word is already explained; demand near-identity
    if len(seed) <= 4:
        return 1.0                # exact only: 'up' scores .80 against 'cup'
    if len(seed) <= 7:
        return 0.85
    if len(part) < 7:
        return 0.85               # short fragment vs long seed: .62 is meaningless
    return 0.62                   # long-vs-long: .62 still shares ~6 chars


def lex_pair(part, seed):
    r = SequenceMatcher(None, part, seed).ratio()
    return r if r >= min_ratio(part, seed) else 0.0


lex_score, lex_seed, lex_dir = [], [], []
for t in list_all_tokens:
    best, bw, bd = 0.0, "", ""
    for part in parts_of(str(t)):
        for w, d in flat_lex:
            r = lex_pair(part, w)
            if r > best:
                best, bw, bd = r, w, d
    lex_score.append(best); lex_seed.append(bw); lex_dir.append(bd)
lex_score = np.array(lex_score)

print(f"{(lex_score > 0).sum()} tokens matched lexically")
for lo, hi in [(0.99, 1.01), (0.85, 0.99), (0.62, 0.85)]:
    idx = np.where((lex_score >= lo) & (lex_score < hi))[0]
    print(f"\n=== [{lo}, {hi}) : {len(idx)} ===")
    print("   ", [repr(str(list_all_tokens[i])) for i in idx[:40]])

119 tokens matched lexically

=== [0.99, 1.01) : 84 ===
    ["'\\tUP'", "'\\tleft'", "'\\tright'", "'\\tup'", "' DOWN'", "' Down'", "' LEFT'", "' Left'", "' RIGHT'", "' Right'", "' UP'", "' Up'", "' abaixo'", "' abajo'", "' acima'", "' derecha'", "' derecho'", "' direita'", "' down'", "' downward'", "' droite'", "' esquerda'", "' gauche'", "' izquierda'", "' left'", "' rechts'", "' right'", "' sopra'", "' up'", "' upward'", "' upwards'", "' вниз'", "'(left'", "'(right'", "'(up'", "',left'", "',right'", "'-UP'", "'-Up'", "'-down'"]

=== [0.85, 0.99) : 12 ===
    ["' derechos'", "' directa'", "' directo'", "' direta'", "' direto'", "' izquier'", "' rechten'", "' rechter'", "'IGHT'", "'ight'", "'unen'", "'uten'"]

=== [0.62, 0.85) : 23 ===
    ["' abierto'", "' bereits'", "' centred'", "' cualquiera'", "' degener'", "' downside'", "' entweder'", "' inexist'", "' infiltr'", "' internas'", "' interno'", "' internos'", "' parfaite'", "' runtime'", "'-industr'", "'Iterable'", "'UInteger'", "'

In [6]:
# 5. embedding model (CPU: the GTX 1050 is sm_61, unsupported by this torch build)
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
_tok = AutoTokenizer.from_pretrained(MODEL)
_mod = AutoModel.from_pretrained(MODEL).eval()


@torch.no_grad()
def encode(texts, batch=256, show_every=None):
    """Mean-pooled, L2-normalised embeddings. Bare strings, no template --
    'the direction "{x}"' compresses everything upward and kills the separation."""
    out = []
    for i in range(0, len(texts), batch):
        b = _tok(texts[i:i + batch], padding=True, truncation=True,
                 max_length=16, return_tensors="pt")
        h = _mod(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).float()
        out.append(F.normalize((h * m).sum(1) / m.sum(1), dim=-1))
        if show_every and (i // batch) % show_every == 0:
            print(f"  {i + len(b['input_ids']):>6d}/{len(texts)}")
    return torch.cat(out).numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
# 6. semantic stage: cosine against the anchors
#
# Two different questions, and they need two different scores.
#
# "Is this direction-related at all?" -> z, standardised per anchor. Raw
# max-cosine fails here because anchors differ wildly in how close they sit to
# the vocabulary at large ('위' averages .622 against random tokens, 'down'
# .329), so a max over raw cosine is really a vote for the hubbiest anchor --
# 8.5k unrelated tokens came back as "up" via '위'.
#
# "Which direction is it?" -> raw cosine. Using z here over-corrects: the token
# '↑' matches anchor '↑' at cos=1.000 but only z=+2.88 (that anchor's
# background is mean .576, std .147), while it matches '→' at a mere cos=.725
# yet z=+3.61 -- so a perfect match lost to a mediocre one and '↑' was filed
# under RIGHT. Cosine is the actual similarity; z is only a calibration for the
# inclusion decision.
anchor_words = [w for ws in SEM_ANCHORS.values() for w in ws]
anchor_dir = [d for d, ws in SEM_ANCHORS.items() for _ in ws]
A = encode(anchor_words)

# many tokens collapse to the same stripped form ('left', ' left', '_left', '.left')
uniq = sorted({strip_token(str(t)) for t in list_all_tokens} - {""})
U = encode(uniq, batch=256, show_every=20)

S = U @ A.T                                             # (n_uniq, n_anchors)
Z = (S - S.mean(0, keepdims=True)) / S.std(0, keepdims=True)

print("anchor hubness (mean cosine to the whole vocab):")
for w, mu in sorted(zip(anchor_words, S.mean(0)), key=lambda x: -x[1])[:5]:
    print(f"  {w:10s} {mu:+.3f}")

i_z = Z.argmax(axis=1)                                  # how strong  -> threshold
i_c = S.argmax(axis=1)                                  # which way   -> label
by_form = {f: (float(Z[i, i_z[i]]), anchor_words[i_c[i]], anchor_dir[i_c[i]])
           for i, f in enumerate(uniq)}

sem_z, sem_seed, sem_dir = [], [], []
for t in list_all_tokens:
    z, w, d = by_form.get(strip_token(str(t)), (-9.0, "", ""))
    sem_z.append(z); sem_seed.append(w); sem_dir.append(d)
sem_z = np.array(sem_z)

for a in ["↑", "↓", "←", "→"]:
    print(f"  {a} -> {by_form[a][2].upper():5s} (via {by_form[a][1]}, z={by_form[a][0]:+.2f})")
print(f"\nembedded {len(uniq)} unique forms for {len(list_all_tokens)} tokens")

     256/17771


    5376/17771


   10496/17771


   15616/17771


anchor hubness (mean cosine to the whole vocab):
  위          +0.621
  giù        +0.603
  sopra      +0.582
  ↑          +0.576
  bas        +0.572
  ↑ -> UP    (via ↑, z=+3.61)
  ↓ -> DOWN  (via ↓, z=+5.39)
  ← -> LEFT  (via ←, z=+3.12)
  → -> RIGHT (via →, z=+6.25)

embedded 17771 unique forms for 28973 tokens


In [8]:
# 7. combine both signals into one frame
LEX_T = 0.01   # the guards in cell 4 already zeroed everything below bar
SEM_T = 3.00   # z, not cosine

df = pd.DataFrame({
    "token": [str(t) for t in list_all_tokens],
    "lex_score": lex_score, "lex_seed": lex_seed, "lex_dir": lex_dir,
    "sem_z": sem_z, "sem_seed": sem_seed, "sem_dir": sem_dir,
})
# cached so SEM_T can be retuned without re-embedding 17.7k forms on CPU
df.to_pickle("/media/alex/D/Uni/northeastern/data/jlens/direction_scores.pkl")

print("semantic z bands (where to put SEM_T):")
for lo, hi in [(5.0, 99), (4.0, 5.0), (3.5, 4.0), (3.0, 3.5), (2.5, 3.0), (2.0, 2.5)]:
    sel = df[(df.sem_z >= lo) & (df.sem_z < hi)]
    print(f"  [{lo:>4}, {hi:>4}) : {len(sel):4d}  {[repr(t) for t in sel.token.head(14)]}")

print(f"\nlexical  : {(df.lex_score >= LEX_T).sum():5d} tokens")
print(f"semantic : {(df.sem_z >= SEM_T).sum():5d} tokens")
print(f"union    : {((df.lex_score >= LEX_T) | (df.sem_z >= SEM_T)).sum():5d} tokens")

semantic z bands (where to put SEM_T):
  [ 5.0,   99) :   88  ["'\\tUP'", "'\\tleft'", "'\\tup'", "' DOWN'", "' Down'", "' Downs'", "' LEFT'", "' Left'", "' UP'", "' Up'", "' Upp'", "' bawah'", "' below'", "' descend'"]
  [ 4.0,  5.0) :   87  ["' Above'", "' BELOW'", "' Below'", "' FALL'", "' Fell'", "' HIGH'", "' Raise'", "' Uph'", "' Upstairs'", "' abajo'", "' above'", "' acima'", "' ascending'", "' augmented'"]
  [ 3.5,  4.0) :   94  ["'\\tright'", "' ABOVE'", "' Drops'", "' HEIGHT'", "' Heights'", "' Higher'", "' Horizontal'", "' Increment'", "' Lose'", "' Loss'", "' Right'", "' Rising'", "' UPC'", "' Upper'"]
  [ 3.0,  3.5) :  175  ["'\\tside'", "' (§'", "' Beyond'", "' Bottom'", "' Collapse'", "' DROP'", "' Deg'", "' EAST'", "' East'", "' Extrem'", "' Here'", "' Ladder'", "' Leave'", "' Leaving'"]
  [ 2.5,  3.0) :  337  ["' Bound'", "' Correct'", "' DEBUG'", "' DEFAULT'", "' Dept'", "' Depth'", "' Direction'", "' Drop'", "' Extreme'", "' FRONT'", "' Following'", "' Forward'", "' 

In [9]:
# 8. final candidate set
review = df[(df.lex_score >= LEX_T) | (df.sem_z >= SEM_T)].copy()
review["hit"] = np.where(
    (review.lex_score >= LEX_T) & (review.sem_z >= SEM_T), "both",
    np.where(review.lex_score >= LEX_T, "lexical", "semantic"),
)
review["direction"] = np.where(review.lex_score >= LEX_T, review.lex_dir, review.sem_dir)
review["rank"] = review[["lex_score", "sem_z"]].max(axis=1)
review = review.sort_values(["direction", "hit", "rank"], ascending=[True, True, False])

print(f"{len(review)} candidates  "
      f"(both={sum(review.hit == 'both')}, "
      f"lexical only={sum(review.hit == 'lexical')}, "
      f"semantic only={sum(review.hit == 'semantic')})")
print(review.direction.value_counts().to_string())

# 'both' is essentially all true positives; the single-signal groups are where
# to spend review effort.
pd.set_option("display.max_rows", 600)
review[["token", "direction", "hit", "lex_score", "lex_seed", "sem_z", "sem_seed"]]

478 candidates  (both=85, lexical only=34, semantic only=359)
direction
up       193
down     173
right     64
left      48


,token,direction,hit,lex_score,lex_seed,sem_z,sem_seed
6351,down,down,both,1.000000,down,6.632871,down
14887,-down,down,both,1.000000,down,6.632871,down
16094,.down,down,both,1.000000,down,6.632871,down
16503,/down,down,both,1.000000,down,6.632871,down
21495,_down,down,both,1.000000,down,6.632871,down
23348,down,down,both,1.000000,down,6.632871,down
1697,Down,down,both,1.000000,down,6.602621,down
15793,.Down,down,both,1.000000,down,6.602621,down
18930,Down,down,both,1.000000,down,6.602621,down
1573,DOWN,down,both,1.000000,down,6.443959,down


In [10]:
# 9. final output: {UP, DOWN, LEFT, RIGHT} -> raw, unprocessed tokens
#
# Everything above (stripping, lowercasing, camel splitting) was scoring
# machinery only. What lands here is the byte-exact vocabulary string --
# leading spaces, tabs, punctuation and all.
import json

direction_tokens = {
    d.upper(): review.loc[review.direction == d, "token"].tolist()
    for d in ["up", "down", "left", "right"]
}

originals = set(str(t) for t in list_all_tokens)
for k, v in direction_tokens.items():
    assert all(t in originals for t in v), f"{k}: token was modified"
assert sum(map(len, direction_tokens.values())) == len(review)
assert len({t for v in direction_tokens.values() for t in v}) == len(review), "token in two directions"

for k, v in direction_tokens.items():
    print(f"{k:6s} {len(v):4d}  {[repr(t) for t in v[:8]]}")

OUT = "/media/alex/D/Uni/northeastern/data/jlens/direction_tokens.json"
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(direction_tokens, f, ensure_ascii=False, indent=2)

# round-trip check: the file must give back byte-identical strings
with open(OUT, encoding="utf-8") as f:
    assert json.load(f) == direction_tokens
print(f"\nwrote {OUT}")

UP      193  ["' upward'", "' upwards'", "'\\tup'", "' up'", "'(up'", "'-up'", "'.up'", "'/up'"]
DOWN    173  ["' down'", "'-down'", "'.down'", "'/down'", "'_down'", "'down'", "' Down'", "'.Down'"]
LEFT     48  ["' esquerda'", "' gauche'", "'\\tleft'", "' left'", "'(left'", "',left'", "'-left'", "'.left'"]
RIGHT    64  ["' rechts'", "' direita'", "' derecha'", "' droite'", "'rightarrow'", "' Right'", "'.Right'", "'Right'"]

wrote /media/alex/D/Uni/northeastern/data/jlens/direction_tokens.json
